In [4]:
!pip install torch transformers onnx onnxruntime fastapi uvicorn pytest onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

# =====================================================
# SIMPLE SENTIMENT MODEL
# =====================================================

class SentimentModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.fc1 = nn.Linear(100, 64)

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):

        x = self.fc1(x)

        x = self.relu(x)

        x = self.fc2(x)

        return x

model = SentimentModel()

print(model)

SentimentModel(
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)


In [3]:
dummy_input = torch.randn(1, 100)

torch.onnx.export(
    model,
    dummy_input,
    "sentiment_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("ONNX Model Exported Successfully!")

ModuleNotFoundError: No module named 'onnxscript'

In [6]:
from fastapi import FastAPI

import onnxruntime as ort

import numpy as np

app = FastAPI()

# =====================================================
# LOAD ONNX MODEL
# =====================================================

session = ort.InferenceSession(
    "sentiment_model.onnx"
)

# =====================================================
# API ENDPOINT
# =====================================================

@app.get("/predict")

def predict():

    sample = np.random.randn(1,100).astype(np.float32)

    outputs = session.run(
        None,
        {"input": sample}
    )

    prediction = np.argmax(outputs[0])

    sentiment = (
        "Positive"
        if prediction == 1
        else "Negative"
    )

    return {
        "prediction": sentiment
    }

In [8]:
!uvicorn app:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [3433] using WatchFiles
ERROR:    Error loading ASGI app. Could not import module "app".
INFO:     Stopping reloader process [3433]


In [12]:
from fastapi.testclient import TestClient

client = TestClient(app)

def test_prediction():

    response = client.get("/predict")

    assert response.status_code == 200

In [5]:
dummy_input = torch.randn(1, 100)

torch.onnx.export(
    model,
    dummy_input,
    "sentiment_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("ONNX Model Exported Successfully!")

/tmp/ipykernel_2375/2420865786.py:3: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/tmp/ipykernel_2375/2420865786.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SentimentModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SentimentModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX Model Exported Successfully!


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
